# Module 1 — Data model and quality

Source conversions are in `scripts/get_data.py`; the model is `sql/01_model.sql`; the builder is `scripts/build_phase1.py`. This notebook reads their saved results. Raw inputs are preserved. Monetary values are USD.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
DATA = ROOT / 'exports/phase1'
def load(name):
    return pd.read_csv(DATA / f'{name}.csv')
manifest = json.loads((ROOT / 'docs/source_manifest.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{'source': f['name'], 'rows': f['rows'], 'sha256': f['sha256']} for f in manifest['files']]))
print('Source commit:', manifest['commit'])

,source,rows,sha256
0,campaign_descriptions.rda,27,601bd17ea1a6de92cc288f393164813fbaf293b997d440...
1,campaigns.rda,6589,33c7b3dc1bc722d465f97416fb1327c6a9a640190e464c...
2,coupon_redemptions.rda,2102,4b68148175ed19300bb2615d68d1a0300a5f971db32408...
3,coupons.rda,116204,4d87effce5b8813c0934b581a83292707347577743320c...
4,demographics.rda,801,8b80455bc841003b64e47f5e5c2221f6093b995e923d59...
5,products.rda,92331,a80c6df33623b4af296ae9f317a6647e369db7e8ce7e7b...
6,promotions.rds,20940529,15a729fdad31b10d3afedb058da31bcd6e9e68cb0207a4...
7,transactions.rds,1469307,1fa0700033f1e5d9bb6b09e2be063d8d68474d346e95c5...


Source commit: 5b5d06192b9856edd04e4d405787af2f2e4a1fef


## 1. Learn the fields and profile the sources

IDs are labels. `sales_value` is retailer receipts; retailer-funded discounts combine loyalty discounts and coupon matching. Manufacturer coupons are reported separately. Inspect nulls before joining or replacing values.

In [2]:
display(load('profile_transaction_summary').T)
columns = load('profile_columns')
display(columns[columns.null_rows > 0])
display(load('profile_relationships'))

,0
line_items,1469307
households,2469
baskets,155848
stores,457
products,68509
first_timestamp_utc,2017-01-01 11:53:26
last_timestamp_utc,2018-01-01 04:01:20
first_week,1
last_week,53
sales_usd,4596039.58


,table_name,column_name,data_type,total_rows,null_rows,distinct_non_null
32,demographics,home_ownership,VARCHAR,801,233,4
33,demographics,marital_status,VARCHAR,801,137,2
41,products,product_category,VARCHAR,92331,540,303
42,products,product_type,VARCHAR,92331,528,2378
43,products,package_size,VARCHAR,92331,30586,3705


,check_name,affected_rows
0,campaigns_without_description,0
1,campaigns_without_purchasing_household,0
2,redemptions_without_campaign,0
3,redemptions_without_coupon_campaign_pair,0
4,coupons_without_product,16
5,promotions_without_product,0
6,duplicate_household_campaign_assignments,0


## 2. Build the model and document cleaning

The model removes only quantity-zero/sales-zero voids. It retains free lines, zero-quantity lines with sales, unknown products and large quantities. Promotion conflicts use MULTIPLE and preserve recorded display/mailer exposure.

In [3]:
display(load('fact_sales_sample').head(10))
display(load('profile_missing_products'))
display(load('profile_promotion_quality'))
display(Markdown((ROOT / 'docs/cleaning_log.md').read_text(encoding='utf-8')))

,product_id,line_items,sales_usd
0,5978656,2932,0.00
1,1093587,25,124.63
2,5978648,1819,0.00
3,5126088,13,0.00
4,5978657,2,0.00
5,5978659,1,0.00
6,926775,5,31.62
7,5126107,5,0.00
8,5977100,3,0.00
9,316011,2,16.32


,raw_rows,unique_keys,extra_rows,repeated_keys,conflicting_keys
0,20940529,20927744,12785,12785,12785


# Cleaning log — verified phase 1

Built 25 October 2026 from the commit-pinned completejourney sources. All eight raw-table counts and content fingerprints matched the original verified setup before and after modelling. Rules below apply to derived tables only. Monetary values are USD; affected sales are the value of the flagged lines, not estimated marketing costs.

| Decision | Actual impact | Sales on affected lines | Reason and evidence |
|---|---:|---:|---|
| Exclude quantity-zero/sales-zero voids from fact_sales | 8,820 lines removed | $0.00 | sql/01_model.sql; sql/00b_rule_impacts.sql |
| Retain free purchase lines | 2,406 retained | $0.00 | Real zero-sales lines can matter for discount depth; not automatically errors |
| Retain zero-quantity lines that have positive sales | 49 retained | $12.52 | No evidence justifies deleting their sales; exclude/define carefully for future unit-price metrics |
| Retain large quantities; inspect their departments | 13,510 retained; max 89,638 | $364,119.46 | FUEL and code-like product quantities are not necessarily individual grocery units; profile_large_quantities records the split |
| Flag non-grocery instead of deleting it | 18,526 clean lines flagged | $410,983.94 | FUEL/MISCELLANEOUS departments or COUPON/MISC ITEMS category; sql/01_model.sql |
| Add UNKNOWN product records | 17 missing product IDs; 4,836 source lines retained | $172.57 | profile_missing_products and dim_product.is_unknown_product |
| Retain discounts greater than price paid | 37,634 lines | $80,407.91 | Do not automatically discard deep discounts |
| Collapse repeated promotion keys | 12,785 extra rows consolidated into 20,927,744 product/store/week rows | Not a sales table | All 12,785 repeated keys have differing placement codes. Flags use any nonzero code; differing labels become MULTIPLE |
| Derive US Eastern local purchase time | All 1,460,487 clean lines; UTC also retained | No sales removed | Explicit UTC-to-America/New_York conversion; profile_week_boundaries checks calendar alignment under the Eastern reporting convention |
| Leave demographics absent where unknown | 801 of 2,469 purchasing households have demographics | No sales removed | No imputation; has_demographics distinguishes observed from missing |
| Flag partial campaigns | 8 of 27 flagged, all retained | No sales removed | Campaigns 15,20,21,22,23,24,25,26; use eligibility rules in later campaign analysis |
| Retain partial week 1 | One observed local date, all source rows retained | No sales removed | Exclude week 1 only in later like-for-like weekly comparisons |
| Review anomaly flags without deleting rows | 10 dates and 142 baskets flagged | No sales removed | sql/01b_anomalies.sql; review exports/phase1/top_five_basket_lines.csv |
| Preserve orphan coupon mappings | 16 coupon rows lack a product metadata match | Not a sales table | profile_relationships; future coupon-product joins must preserve or explicitly report them |

## Reconciliation

- Source transaction lines: 1,469,307; clean sales lines: 1,460,487. Difference: 8,820 voids.
- Source and clean sales both equal $4,596,039.58 to the cent.
- Grocery: 1,441,961 lines and $4,185,055.64. Non-grocery: 18,526 lines and $410,983.94. UNKNOWN products are retained within the not-explicitly-non-grocery population; do not imply their category has been identified.
- No negative quantity/sales or negative discount rows were observed. Required source fields are non-null; nullable metadata remains nullable.
- 24 database verification rules passed. No sales were removed for anomaly flags, campaign coverage, missing demographics, or unknown products.

The raw source rows are preserved in grocery_phase1.duckdb as well as the original raw-only grocery.duckdb. Diagnostic CSVs and the saved SQL make the decisions reproducible.


## 3. Review anomalies and time boundaries

Flags prompt inspection, not deletion. Local dates use America/New_York; UTC timestamps are retained. A missing purchase date does not prove a store was closed.

In [4]:
display(load('daily_anomalies'))
display(load('basket_anomalies').sort_values('basket_value', ascending=False).head(10))
display(load('profile_missing_days'))
display(load('profile_campaign_windows'))

,day,weekday,sales,baskets,typical_for_weekday,weekday_mad,robust_z
0,2017-01-01,Sunday,10106.85,369,14760.410,927.980,-3.382382
1,2017-02-04,Saturday,18250.25,473,13346.370,703.600,4.700997
2,2017-04-15,Saturday,17041.61,472,13346.370,703.600,3.542361
3,2017-07-04,Tuesday,16090.84,495,9962.710,681.960,6.061011
4,2017-11-04,Saturday,16801.76,476,13346.370,703.600,3.312434
5,2017-11-21,Tuesday,16102.90,450,9962.710,681.960,6.072939
6,2017-11-22,Wednesday,18859.07,597,9622.225,611.630,10.186169
7,2017-11-25,Saturday,9026.03,318,13346.370,703.600,-4.141599
8,2017-12-22,Friday,17333.65,426,10763.300,952.215,4.654033
9,2017-12-23,Saturday,22954.80,580,13346.370,703.600,9.210911


,day
0,2017-12-25


,campaign_id,campaign_type,start_date,end_date,partially_observed,invalid_window
0,1,Type B,2017-03-03,2017-04-09,False,False
1,2,Type B,2017-03-08,2017-04-09,False,False
2,3,Type C,2017-03-13,2017-05-08,False,False
3,4,Type B,2017-03-29,2017-04-30,False,False
4,5,Type B,2017-04-03,2017-05-07,False,False
5,6,Type C,2017-04-19,2017-05-21,False,False
6,7,Type B,2017-04-24,2017-05-28,False,False
7,8,Type A,2017-05-08,2017-06-25,False,False
8,9,Type B,2017-05-31,2017-07-02,False,False
9,10,Type B,2017-06-28,2017-07-30,False,False


## 4. Reconcile the model

Every check must pass. To explore in SQL, open `grocery_phase1.duckdb` in any SQL client and run `sql/01d_inspect.sql`; use Ctrl+Enter for one query at a time. The builder reproduces the module in a fresh project copy.

In [5]:
checks = load('phase1_checks')
assert checks.passed.astype(str).str.lower().eq('true').all()
print(f'{len(checks)} model checks passed')
display(checks)
display(Markdown((ROOT / 'docs/phase1_findings.md').read_text(encoding='utf-8')))

24 model checks passed


,check_name,passed
0,sales_rows_reconcile,True
1,sales_money_reconciles,True
2,no_void_lines,True
3,free_lines_preserved,True
4,product_keys_unique,True
5,household_keys_unique,True
6,campaign_keys_unique,True
7,sales_products_covered,True
8,sales_households_covered,True
9,promotion_rows_reconcile,True


# Phase 1 findings

## What is complete

Eight full raw datasets, all-column source profiling, clean sales and promotion models, three dimensions, a campaign-recipient bridge, anomaly screening, exported diagnostics, and verification evidence. The derived model's six main table counts match the expected totals. The working database is grocery_phase1.duckdb.

## Verified results

| Measure | Actual result |
|---|---:|
| Raw purchase lines | 1,469,307 |
| Clean purchase lines | 1,460,487 |
| Purchasing households | 2,469 |
| Baskets | 155,848 |
| Stores | 457 |
| Products appearing in purchases | 68,509 |
| Recorded sales, preserved after cleaning | $4,596,039.58 |
| Product dimension, including UNKNOWN additions | 92,348 |
| Promotion rows after consolidation | 20,927,744 |
| Conflicting promotion keys | 12,785 |
| Observed grocery purchase dates in local 2017 | 364 |
| Flagged dates / grocery baskets | 10 / 142 |
| Database checks | 24 passed |

## Findings that affect later analyses

1. **Purchases must not disappear when product metadata is missing.** Seventeen product IDs account for 4,836 lines and $172.57. They now have UNKNOWN dimension records.
2. **Promotion duplicates contain conflicting information.** All 12,785 extra rows belong to keys with different placement codes. Exposure flags preserve whether any row shows a display/mailer; MULTIPLE labels expose the ambiguity for placement-specific analysis.
3. **Quantity needs context.** There are 13,510 quantities above 100, with a maximum of 89,638. Review their departments before treating quantity as a count of grocery units. Forty-nine lines have positive sales but zero quantity, so unit-price calculations need an explicit eligibility rule.
4. **Metadata coverage is incomplete.** Only 801 purchasing households have demographics. There are also 16 coupon-product mapping rows without a product metadata match. Both facts are preserved and reported.
5. **Time windows matter.** The source timestamps span 2017-01-01 11:53:26 to 2018-01-01 04:01:20 UTC, with source weeks 1–53. Convert to local time before date-based analysis. Week 1 has one observed local date. Eight campaigns cross the calendar observation window.
6. **Missing dates are not explanations.** The local grocery purchase calendar has no records for 2017-12-25. That alone does not prove store closure. Anomaly flags are review prompts, not confirmed bad records or causal findings.

## Source missingness

Required modelling fields were non-null. These metadata columns contain missing values:

| Table | Column | Null rows |
|---|---|---:|
| demographics | home_ownership | 233 |
| demographics | marital_status | 137 |
| products | package_size | 30,586 |
| products | product_category | 540 |
| products | product_type | 528 |

All 44 source columns have row counts, missingness, cardinality, and types in profile_columns and its CSV export.

## Flagged daily sales

Grocery-only sales, calculated in local time. Scores compare each date with the median for its weekday, using median absolute deviation. Possible holiday explanations are hypotheses, not verified facts.

| Date | Robust z-score |
|---|---:|
| 2017-11-22 | +10.19 |
| 2017-12-23 | +9.21 |
| 2017-11-21 | +6.07 |
| 2017-07-04 | +6.06 |
| 2017-02-04 | +4.70 |
| 2017-12-22 | +4.65 |
| 2017-04-15 | +3.54 |
| 2017-11-04 | +3.31 |
| 2017-01-01 | -3.38 |
| 2017-11-25 | -4.14 |

The basket export flags 142 grocery baskets above the 99.9th percentile. Line-level records for the five highest flagged baskets are provided for inspection; flags do not automatically remove them.

## Verification and limits

- 24 checks cover source immutability, row and monetary reconciliation, dimension uniqueness, fact coverage, promotion grain, conflict accounting, discount formulas, local-time conversion, campaign dates, required-field missingness, and anomaly outputs.
- The synthetic model regression covers free purchases, unknown products, nullable product attributes, non-grocery classification, conflicting promotions, duplicate campaign recipients, and time conversion.
- The six reader regressions compare the compact reader with the standard reader on the six smaller R sources.
- Later modules cover KPIs, segments, promotions, campaigns and the email experiment.
